In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, avg, sum, min, max,
    hour, dayofweek, to_timestamp,
    percentile_approx, round
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType,
    DoubleType, StringType, TimestampType
)

In [28]:
# 1. CREATE SPARK SESSION
spark = (
    SparkSession.builder
    .appName("NYC Taxi Big Data Analytics")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

ConnectionRefusedError: [Errno 61] Connection refused

In [3]:
input_path = "data/yellow_tripdata_2025-01.parquet"

df = spark.read.parquet(input_path)

print("Initial Records:", df.count())

print("Initial Schema:")
df.printSchema()


Initial Records: 3475226
Initial Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



26/11/15 22:10:28 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
# 3. SELECT RELEVANT COLUMNS
df = df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount"
)

In [5]:
# 4. DATA QUALITY ANALYSIS
# -------------------------------------------------------

print("\nMissing Values:")

missing = df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

missing.show()



Missing Values:


+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+
|       0|                   0|                    0|         540149|            0|           0|           0|           0|          0|         0|           0|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+



In [6]:
df = df.dropDuplicates()

print("Records after duplicate removal:", df.count())


# -------------------------------------------------------
# 6. HANDLE MISSING VALUES
# -------------------------------------------------------

df = df.fillna({
    "passenger_count": 1,
    "trip_distance": 0.0,
    "fare_amount": 0.0,
    "tip_amount": 0.0,
    "total_amount": 0.0
})



26/11/15 22:10:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:49 WARN RowBasedKeyValueBatch: Calling spill() on

Records after duplicate removal: 3475226


In [7]:
# 7. REMOVE INVALID RECORDS
# -------------------------------------------------------

df = df.filter(
    (col("trip_distance") >= 0) &
    (col("fare_amount") >= 0) &
    (col("total_amount") >= 0) &
    (col("passenger_count") > 0)
)

In [8]:
# 8. OUTLIER DETECTION
# -------------------------------------------------------

quantiles = df.select(
    percentile_approx("trip_distance", 0.99).alias("p99_distance"),
    percentile_approx("fare_amount", 0.99).alias("p99_fare")
).collect()[0]

p99_distance = quantiles["p99_distance"]
p99_fare = quantiles["p99_fare"]

print("99th percentile trip distance:", p99_distance)
print("99th percentile fare:", p99_fare)


26/11/15 22:10:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 22:10:58 WARN RowBasedKeyValueBatch: Calling spill() on

99th percentile trip distance: 19.52
99th percentile fare: 73.7


In [9]:
# 9. REMOVE EXTREME OUTLIERS
# -------------------------------------------------------

df = df.filter(
    (col("trip_distance") <= p99_distance) &
    (col("fare_amount") <= p99_fare)
)

In [10]:
from pyspark.sql.functions import (
    col, hour, dayofweek, to_timestamp
)

df = (
    df
    .withColumn(
        "pickup_time",
        to_timestamp(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "dropoff_time",
        to_timestamp(col("tpep_dropoff_datetime"))
    )
    .withColumn(
        "pickup_hour",
        hour(col("pickup_time"))
    )
    .withColumn(
        "day_of_week",
        dayofweek(col("pickup_time"))
    )
    .withColumn(
        "trip_duration_minutes",
        (
            col("dropoff_time").cast("long") -
            col("pickup_time").cast("long")
        ) / 60.0
    )
)

In [11]:
df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_time",
    "dropoff_time",
    "pickup_hour",
    "day_of_week",
    "trip_duration_minutes"
).show(10, truncate=False)

+--------------------+---------------------+-------------------+-------------------+-----------+-----------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|pickup_time        |dropoff_time       |pickup_hour|day_of_week|trip_duration_minutes|
+--------------------+---------------------+-------------------+-------------------+-----------+-----------+---------------------+
|2025-01-01 00:17:57 |2025-01-01 00:36:57  |2025-01-01 00:17:57|2025-01-01 00:36:57|0          |4          |19.0                 |
|2025-01-01 00:37:09 |2025-01-01 00:51:05  |2025-01-01 00:37:09|2025-01-01 00:51:05|0          |4          |13.933333333333334   |
|2025-01-01 00:32:52 |2025-01-01 01:13:47  |2025-01-01 00:32:52|2025-01-01 01:13:47|0          |4          |40.916666666666664   |
|2025-01-01 00:22:17 |2025-01-01 00:46:13  |2025-01-01 00:22:17|2025-01-01 00:46:13|0          |4          |23.933333333333334   |
|2025-01-01 00:02:49 |2025-01-01 00:14:48  |2025-01-01 00:02:49|2025-01-01 00:14:48

In [12]:
# 11. REMOVE IMPOSSIBLE TRIP DURATIONS
# -------------------------------------------------------

df = df.filter(
    (col("trip_duration_minutes") > 0) &
    (col("trip_duration_minutes") <= 300)
)


In [13]:
print("\n===== BASIC STATISTICS =====")

df.select(
    round(avg("trip_distance"), 2).alias("avg_distance"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(avg("total_amount"), 2).alias("avg_total"),
    round(avg("trip_duration_minutes"), 2).alias("avg_duration")
).show()


===== BASIC STATISTICS =====


+------------+--------+---------+------------+
|avg_distance|avg_fare|avg_total|avg_duration|
+------------+--------+---------+------------+
|        2.85|   16.95|    25.52|       14.17|
+------------+--------+---------+------------+



In [23]:
print("\n===== TRIPS BY HOUR =====")

hourly = (
    df.groupBy("pickup_hour")
      .agg(
          count("*").alias("total_trips"),
          round(avg("fare_amount"), 2).alias("avg_fare"),
          round(avg("trip_distance"), 2).alias("avg_distance")
      )
      .orderBy("pickup_hour")
)

hourly.show(24)



===== TRIPS BY HOUR =====


ConnectionRefusedError: [Errno 61] Connection refused

In [22]:
# 14. TOP PICKUP LOCATIONS
# -------------------------------------------------------

print("\n===== TOP PICKUP LOCATIONS =====")

top_pickup = (
    df.groupBy("PULocationID")
      .agg(
          count("*").alias("trips"),
          round(sum("total_amount"), 2).alias("revenue")
      )
      .orderBy(col("trips").desc())
)

top_pickup.show(20)



===== TOP PICKUP LOCATIONS =====


ConnectionRefusedError: [Errno 61] Connection refused

In [21]:
# 15. PAYMENT METHOD ANALYSIS
# -------------------------------------------------------

print("\n===== PAYMENT METHOD ANALYSIS =====")

payment_analysis = (
    df.groupBy("payment_type")
      .agg(
          count("*").alias("transactions"),
          round(avg("fare_amount"), 2).alias("avg_fare"),
          round(avg("tip_amount"), 2).alias("avg_tip"),
          round(sum("total_amount"), 2).alias("revenue")
      )
      .orderBy(col("revenue").desc())
)

payment_analysis.show()




===== PAYMENT METHOD ANALYSIS =====


ConnectionRefusedError: [Errno 61] Connection refused

In [20]:
# 16. PASSENGER ANALYSIS
# -------------------------------------------------------

print("\n===== PASSENGER ANALYSIS =====")

passenger_analysis = (
    df.groupBy("passenger_count")
      .agg(
          count("*").alias("trips"),
          round(avg("trip_distance"), 2).alias("avg_distance"),
          round(avg("fare_amount"), 2).alias("avg_fare")
      )
      .orderBy("passenger_count")
)

passenger_analysis.show()


===== PASSENGER ANALYSIS =====


ConnectionRefusedError: [Errno 61] Connection refused

In [23]:
# 17. DAILY REVENUE
# -------------------------------------------------------

print("\n===== DAILY REVENUE =====")

daily_revenue = (
    df.groupBy(
        df.tpep_pickup_datetime.cast("date").alias("date")
    )
    .agg(
        count("*").alias("trips"),
        round(sum("total_amount"), 2).alias("revenue"),
        round(avg("fare_amount"), 2).alias("avg_fare")
    )
    .orderBy("date")
)

daily_revenue.show(31)


===== DAILY REVENUE =====


26/10/01 06:56:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/01 06:56:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/01 06:56:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/01 06:56:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/01 06:56:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/01 06:56:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/01 06:56:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/01 06:56:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/01 06:56:38 WARN RowBasedKeyValueBatch: Calling spill() on

+----------+------+----------+--------+
|      date| trips|   revenue|avg_fare|
+----------+------+----------+--------+
|2024-12-31|    21|    589.17|   19.12|
|2025-01-01| 79725| 2120790.4|   18.57|
|2025-01-02| 79170| 2162996.8|    18.5|
|2025-01-03| 85922|2259303.46|   17.66|
|2025-01-04| 92236|2321403.39|   17.29|
|2025-01-05| 74467|2028843.08|   18.33|
|2025-01-06| 75643|2040310.73|   17.64|
|2025-01-07| 95185|2436577.15|   16.57|
|2025-01-08|107263|2700502.37|   16.33|
|2025-01-09|112007|2852737.62|   16.57|
|2025-01-10|104905|2709243.63|   16.86|
|2025-01-11|114719|2764614.11|   16.08|
|2025-01-12| 95353|2450086.29|    17.3|
|2025-01-13| 94392|2464890.82|   17.02|
|2025-01-14|114433| 2904624.8|    16.5|
|2025-01-15|118206|3013924.58|   16.65|
|2025-01-16|131101|3398872.19|   17.12|
|2025-01-17|112790|2926279.87|   17.14|
|2025-01-18|111784|2657657.08|   16.05|
|2025-01-19|108425|2568184.32|   16.41|
|2025-01-20| 82446|2202981.64|   18.26|
|2025-01-21|110037|2822977.19|   16.96|


In [24]:
# 18. CACHE FOR REPEATED ANALYSIS
# -------------------------------------------------------

df.cache()

print("Final cleaned records:", df.count())



Final cleaned records: 3251071


In [25]:
output_path = "output/cleaned_taxi_data"

(
    df.write
      .mode("overwrite")
      .parquet(output_path)
)


26/10/01 06:57:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/01 06:57:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/01 06:57:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/10/01 06:57:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/10/01 06:57:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/10/01 06:57:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/10/01 06:57:14 WARN MemoryManager: Total allocation exceeds 95.

In [26]:
# 20. SAVE ANALYTICS RESULTS
# -------------------------------------------------------

hourly.write.mode("overwrite").option(
    "header", "true"
).csv("output/hourly_analysis")

top_pickup.write.mode("overwrite").option(
    "header", "true"
).csv("output/top_pickup_locations")

payment_analysis.write.mode("overwrite").option(
    "header", "true"
).csv("output/payment_analysis")

daily_revenue.write.mode("overwrite").option(
    "header", "true"
).csv("output/daily_revenue")

In [27]:
# 21. STOP SPARK
# -------------------------------------------------------

spark.stop()